In [46]:
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision.io import decode_image
import torch.nn.functional as F
from torchvision.datasets import ImageFolder
from torchvision.transforms import v2, InterpolationMode
from PIL import Image

In [3]:

device = "cuda" if torch.cuda.is_available() else "cpu"


In [47]:
# DINO v2 transform: https://github.com/facebookresearch/dinov3
def make_transform(resize_size: int=280):
    to_tensor = v2.ToImage()
    resize = v2.Resize((resize_size, resize_size), antialias=True)
    to_float = v2.ToDtype(torch.float32, scale=True)
    normalize = v2.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    )
    return v2.Compose([to_tensor, resize, to_float, normalize])

root = Path('data')
image_folder = ImageFolder(root / "train_images")


samples = [
    (Path(path), class_idx)
    for path, class_idx in image_folder.samples
]

class_names = image_folder.classes

class ForgeryDataset(Dataset):
    def __init__(self, samples, mask_dir: Path | None=None, size=280):
        self.samples = samples
        self.mask_dir = mask_dir

        self.size = size

        self.image_transforms = make_transform(self.size)

        self.mask_transforms = v2.Compose([
            v2.Resize((self.size, self.size), 
            interpolation=InterpolationMode.NEAREST),
        ])
    
    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]

        image = Image.open(img_path).convert("RGB")
        image = self.image_transforms(image)

        if self.mask_dir is not None and "forged" in img_path.parent.name:
            mask_path = self.mask_dir / img_path.name.replace(".png", ".npy")
            mask = np.load(mask_path)
            mask = torch.from_numpy(mask)
            if mask.ndim == 2:
                mask = mask.unsqueeze(0)
            mask = self.mask_transforms(mask)
            mask = mask.squeeze(0).long()
        else:
            mask = torch.from_numpy(np.zeros((256,  256)))

        return image, mask, label

train_image_folder = ImageFolder(root / "train_images")

samples = [(Path(p), y) for p, y in train_image_folder.samples]

train_dataset = ForgeryDataset(
    samples=samples,
    mask_dir=root / "train_masks",
)

In [48]:
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=False)

## Model

In [50]:
model_name = "dinov2_vits14" # small version for testing, replace later?
model= torch.hub.load(
    "facebookresearch/dinov2", model_name, pretrained=True
)
model.eval()
model = model.to(device)

Using cache found in /home/maxba/.cache/torch/hub/facebookresearch_dinov2_main


In [51]:
model

DinoVisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 384, kernel_size=(14, 14), stride=(14, 14))
    (norm): Identity()
  )
  (blocks): ModuleList(
    (0-11): 12 x NestedTensorBlock(
      (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
      (attn): MemEffAttention(
        (qkv): Linear(in_features=384, out_features=1152, bias=True)
        (proj): Linear(in_features=384, out_features=384, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): LayerScale()
      (drop_path1): Identity()
      (norm2): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=384, out_features=1536, bias=True)
        (act): GELU(approximate='none')
        (fc2): Linear(in_features=1536, out_features=384, bias=True)
        (drop): Dropout(p=0.0, inplace=False)
      )
      (ls2): LayerScale()
      (drop_path2): Identity()
    )
  )
  (norm): LayerNorm((384,), eps=1e-06, elementwise_affi

In [52]:
embeddings = []
labels = []

with torch.no_grad():
    for images, masks, label in train_loader:
        images = images.to(device)
        # get image embeddings
        feats = model(images)  # shape: [batch_size, embed_dim]
        embeddings.append(feats.cpu())
        labels.append(label)

embeddings = torch.cat(embeddings)
labels = torch.cat(labels)
print(embeddings.shape)

RuntimeError: stack expects each tensor to be equal size, but got [256, 256] at entry 0 and [280, 280] at entry 9

In [ ]:

# CREDIT: https://www.kaggle.com/code/gauravparkhedkar/dinov2-base-0-332-high-res-4500px-robust-inf

class DinoTinyDecoder(nn.Module):
    def __init__(self, in_ch=768, out_ch=1):
        super().__init__()
        self.block1 = nn.Sequential(nn.Conv2d(in_ch, 384, 3, 1, 1), nn.ReLU(True), nn.Dropout2d(0.1))
        self.block2 = nn.Sequential(nn.Conv2d(384, 192, 3, 1, 1), nn.ReLU(True), nn.Dropout2d(0.1))
        self.block3 = nn.Sequential(nn.Conv2d(192, 96, 3, 1, 1), nn.ReLU(True))
        self.conv_out = nn.Conv2d(96, out_ch, 1)
    
    def forward(self, f, target_size):
        x = F.interpolate(self.block1(f), size=(74, 74), mode='bilinear', align_corners=False)
        x = F.interpolate(self.block2(x), size=(148, 148), mode='bilinear', align_corners=False)
        x = F.interpolate(self.block3(x), size=(296, 296), mode='bilinear', align_corners=False)
        x = self.conv_out(x)
        return F.interpolate(x, size=target_size, mode='bilinear', align_corners=False)

class DinoSegmenter(nn.Module):
    def __init__(self, encoder, processor):
        super().__init__()
        self.encoder, self.processor = encoder, processor
        self.seg_head = DinoTinyDecoder(768, 1)
        
    def forward_features(self, x):
        imgs = (x * 255).clamp(0, 255).byte().permute(0, 2, 3, 1).cpu().numpy()
        inputs = self.processor(images=list(imgs), return_tensors="pt").to(x.device)
        feats = self.encoder(**inputs).last_hidden_state
        B, N, C = feats.shape
        fmap = feats[:, 1:, :].permute(0, 2, 1)
        s = int(np.sqrt(N - 1))
        fmap = fmap.reshape(B, C, s, s)
        return fmap
        
    def forward_seg(self, x):
        fmap = self.forward_features(x)
        return self.seg_head(fmap, (518, 518))